In [1]:
# %% 
# Embedding Table Builder
# -----------------------
# For every token in vocab, get its MiniLM 384-dim embedding.
# All tokens treated equally — special tokens included.
# Saves (vocab_size, 384) float32 array to disk.

import numpy as np
import torch
from tokenizers import Tokenizer
from sentence_transformers import SentenceTransformer

# %%
# Load tokenizer
tokenizer = Tokenizer.from_file("tokenizer/tokenizer.json")
vocab = tokenizer.get_vocab()             # {token_str: id}
id_to_token = {v: k for k, v in vocab.items()}
vocab_size = tokenizer.get_vocab_size()

print(f"Vocab size: {vocab_size}")
print(f"Sample tokens: {list(id_to_token.items())[:10]}")

# %%
# Load MiniLM
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model.eval()

print("MiniLM loaded")

# %%
# Collect all token surface strings in ID order
# ByteLevel tokens have Ġ (space) prefix — decode to real text
token_strings = []
for i in range(vocab_size):
    surface = id_to_token[i]
    surface = surface.replace("Ġ", " ").replace("Ċ", "\n")
    token_strings.append(surface)

print(f"Total tokens to embed: {len(token_strings)}")
print(f"Sample surfaces: {token_strings[:17]}")   # show special tokens too

# %%
# Embed all tokens in batches
BATCH_SIZE = 256
embeddings = []

for i in range(0, len(token_strings), BATCH_SIZE):
    batch = token_strings[i : i + BATCH_SIZE]
    with torch.no_grad():
        vecs = model.encode(
            batch,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    embeddings.append(vecs)
    print(f"  {min(i + BATCH_SIZE, vocab_size)}/{vocab_size}", end="\r")

embedding_table = np.vstack(embeddings).astype(np.float32)
print(f"\nEmbedding table shape: {embedding_table.shape}")   # (vocab_size, 384)

# %%
# Quick sanity check — similar tokens should be close in embedding space
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

cat_id  = vocab.get("cat",  vocab.get("Ġcat",  None))
dog_id  = vocab.get("dog",  vocab.get("Ġdog",  None))
king_id = vocab.get("king", vocab.get("Ġking", None))
pad_id  = vocab.get("<pad>")
unk_id  = vocab.get("<unk>")
bos_id  = vocab.get("<bos>")

if cat_id and dog_id:
    sim = cosine_sim(embedding_table[cat_id], embedding_table[dog_id])
    print(f"Similarity cat  <-> dog  : {sim:.4f}  (expect high ~0.7+)")

if cat_id and king_id:
    sim = cosine_sim(embedding_table[cat_id], embedding_table[king_id])
    print(f"Similarity cat  <-> king : {sim:.4f}  (expect lower)")

# Special tokens should be distinct from each other
sim_pad_unk = cosine_sim(embedding_table[pad_id], embedding_table[unk_id])
sim_pad_bos = cosine_sim(embedding_table[pad_id], embedding_table[bos_id])
print(f"Similarity <pad> <-> <unk>: {sim_pad_unk:.4f}  (distinct, not zero)")
print(f"Similarity <pad> <-> <bos>: {sim_pad_bos:.4f}  (distinct, not zero)")

# %%
# Save
np.save("tokenizer/token_embeddings.npy", embedding_table)
print(f"Saved → tokenizer/token_embeddings.npy")
print(f"Size  : {embedding_table.nbytes / 1024 / 1024:.1f} MB")

Vocab size: 4096
Sample tokens: [(2753, 'Ġtomato'), (899, 'Ġkid'), (2835, 'Ġjet'), (829, 'udden'), (27, '5'), (2154, 'ĠJust'), (1778, 'Ġlonely'), (1326, 'aby'), (552, 'dd'), (1161, 'Ġmaking')]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MiniLM loaded
Total tokens to embed: 4096
Sample surfaces: ['<pad>', '<unk>', '<bos>', '<eos>', '<|system|>', '<|user|>', '<|assistant|>', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*']
  4096/4096
Embedding table shape: (4096, 384)
Similarity cat  <-> dog  : 0.6606  (expect high ~0.7+)
Similarity cat  <-> king : 0.3610  (expect lower)
Similarity <pad> <-> <unk>: 0.5141  (distinct, not zero)
Similarity <pad> <-> <bos>: 0.5118  (distinct, not zero)
Saved → tokenizer/token_embeddings.npy
Size  : 6.0 MB


### Compressor — 384 → 64
#### Trainable MLP that squeezes MiniLM embeddings down to 64-dim
#### This is what the LLM will actually see


In [2]:

import torch
import torch.nn as nn
import numpy as np

# %%
# Load embedding table
embedding_table = np.load("tokenizer/token_embeddings.npy")
embedding_table = torch.tensor(embedding_table, dtype=torch.float32)

print(f"Embedding table: {embedding_table.shape}")  # (4096, 384)

# %%
# Define compressor
class Compressor(nn.Module):
    def __init__(self, in_dim=384, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.LayerNorm(out_dim),
        )

    def forward(self, x):
        # x: (batch, seq_len, 384) → (batch, seq_len, 64)
        return self.net(x)

compressor = Compressor()
print(f"Compressor params: {sum(p.numel() for p in compressor.parameters()):,}")

# %%
# Quick test — pass a few token embeddings through
sample_ids = torch.tensor([0, 1, 2, 3, 4])           # first 5 tokens
sample_emb = embedding_table[sample_ids]              # (5, 384)
sample_emb = sample_emb.unsqueeze(0)                  # (1, 5, 384) — fake batch dim

out = compressor(sample_emb)
print(f"Input  shape: {sample_emb.shape}")            # (1, 5, 384)
print(f"Output shape: {out.shape}")                   # (1, 5, 64)

Embedding table: torch.Size([4096, 384])
Compressor params: 24,768
Input  shape: torch.Size([1, 5, 384])
Output shape: torch.Size([1, 5, 64])
